<a href="https://colab.research.google.com/github/magnogomes874/SSD-/blob/main/MVP2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MVP — Planejamento de uma nova unidade em nuvem**

# **Magno Gomes Sousa - 200031031**


Comparamos custo, prazo e risco de três cenários usando a Multi-Cloud Resource Dataset.


**1. Bibliotecas e importação**

Envie um CSV ou ZIP contendo um único CSV.

In [1]:
import io, math, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print
try:
    import ipywidgets as widgets
except ImportError:
    widgets = None
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 3)
pd.set_option('display.max_colwidth', 120)
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 10})
print('Tudo pronto para começar!')


Tudo pronto para começar!


In [2]:

CAMINHO_BASE = None
if CAMINHO_BASE is None:
    from google.colab import files
    uploaded = files.upload()
else:
    uploaded = {Path(CAMINHO_BASE).name: Path(CAMINHO_BASE).read_bytes()}
if len(uploaded) != 1:
    raise ValueError("Envie somente um CSV ou ZIP.")
nome_arquivo, conteudo = next(iter(uploaded.items()))
if nome_arquivo.lower().endswith(".zip"):
    with zipfile.ZipFile(io.BytesIO(conteudo)) as z:
        csvs = [p for p in z.namelist() if p.lower().endswith(".csv")
                and not p.startswith("__MACOSX/")]
        if len(csvs) != 1:
            raise ValueError(f"Selecione um único CSV. Encontrados: {csvs}")
        print("CSV encontrado:", csvs[0])
        conteudo = z.read(csvs[0])
elif not nome_arquivo.lower().endswith(".csv"):
    raise ValueError("O arquivo deve ser CSV ou ZIP.")
try:
    df_original = pd.read_csv(io.BytesIO(conteudo), dtype="string", encoding="utf-8-sig")
except UnicodeDecodeError:
    df_original = pd.read_csv(io.BytesIO(conteudo), dtype="string", encoding="latin1")
print(f"Base carregada: {len(df_original)} registros e {len(df_original.columns)} colunas.")
display(df_original.head())


Saving archive (1).zip to archive (1).zip
CSV encontrado: Cloud_Dataset.csv
Base carregada: 1000 registros e 16 colunas.


,timestamp,cpu_usage,memory_usage,net_io,disk_io,cloud_provider,region,vm_type,vCPU,RAM_GB,price_per_hour,target,latency_ms,throughput,cost,utilization
0,2024-01-01 00:00:00,43.71,95.56,379.4,638.79,Azure,us-east,t2.micro,1,1.0,0.0116,scale_up,228.02,1380.99,0.0174,69.64
1,2024-01-01 00:05:00,15.23,87.96,320.5,737.27,Azure,us-west,t2.micro,1,1.0,0.0116,scale_up,243.71,1059.03,0.0174,51.59
2,2024-01-01 00:10:00,26.36,26.51,186.91,572.28,AWS,us-east,B1s,1,1.0,0.012,scale_down,256.32,500.59,0.018,26.44
3,2024-01-01 00:15:00,65.07,22.55,181.47,429.73,AWS,asia-south,B1s,1,1.0,0.012,no_action,270.85,864.91,0.018,43.81
4,2024-01-01 00:20:00,27.97,56.28,316.59,141.81,Azure,us-east,B1s,1,1.0,0.012,no_action,246.37,861.28,0.018,42.12


**2. Limpeza da base**

Preserva o original, padroniza campos e valida números, datas e percentuais. Separa registros inválidos para revisão.


In [3]:
import re
import unicodedata
import numpy as np
import pandas as pd

def limpar_base(base_original):
    """Retorna base limpa, registros separados, resumo e relatório por coluna.
    Não modifica o DataFrame original e não inventa valores ausentes.
    """
    df = base_original.copy(deep=True)
    if not df.columns.is_unique:
        raise ValueError("O arquivo possui nomes de colunas repetidos.")
    nomes = [re.sub(r"[^a-z0-9]+", "_", unicodedata.normalize("NFKD", str(c))
             .encode("ascii", "ignore").decode().strip().lower()).strip("_")
             for c in df.columns]
    if len(set(nomes)) != len(nomes):
        raise ValueError("A padronização produz nomes repetidos. Revise o cabeçalho.")
    df.columns = nomes
    numericas = ["cpu_usage", "memory_usage", "net_io", "disk_io", "vcpu",
                 "ram_gb", "price_per_hour", "latency_ms", "throughput",
                 "cost", "utilization"]
    textuais = ["cloud_provider", "region", "vm_type", "target"]
    obrigatorias = ["timestamp"] + numericas + textuais
    faltantes = sorted(set(obrigatorias) - set(df.columns))
    if faltantes:
        raise ValueError(f"Colunas ausentes: {faltantes}. Use o CSV Multi-Cloud.")
    # Índice corresponde ao número do registro no arquivo, começando em 1.
    df.index = pd.RangeIndex(1, len(df) + 1, name="registro_origem")
    for c in df.select_dtypes(include=["object", "string"]).columns:
        df[c] = df[c].astype("string").str.strip().replace("", pd.NA)
    motivos = pd.Series("", index=df.index, dtype="string")
    def sinalizar(mascara, motivo):
        mascara = pd.Series(mascara, index=df.index).fillna(False)
        motivos.loc[mascara] = motivos.loc[mascara] + motivo + "; "
    vazias = df.isna().all(axis=1)
    sinalizar(vazias, "Linha totalmente vazia")
    duplicadas = df.duplicated(keep="first") & ~vazias
    sinalizar(duplicadas, "Registro duplicado após remoção de espaços")
    antes = df.isna().sum()
    falhas = {}
    for c in numericas:
        original = df[c].copy()
        df[c] = pd.to_numeric(df[c], errors="coerce")
        falhas[c] = int((original.notna() & df[c].isna()).sum())
        sinalizar(original.notna() & df[c].isna(), f"Número inválido em {c}")
        infinito = df[c].notna() & ~np.isfinite(df[c].astype(float))
        sinalizar(infinito, f"Número infinito em {c}")
        df.loc[infinito, c] = np.nan
    original = df["timestamp"].copy()
    df["timestamp"] = pd.to_datetime(original, format="%Y-%m-%d %H:%M:%S", errors="coerce")
    falhas["timestamp"] = int((original.notna() & df.timestamp.isna()).sum())
    sinalizar(original.notna() & df.timestamp.isna(), "Data inválida")
    df["cloud_provider"] = df.cloud_provider.str.casefold().replace(
        {"aws": "AWS", "azure": "Azure", "gcp": "GCP"})
    for c in ["region", "vm_type", "target"]:
        df[c] = df[c].str.lower()
    for c in obrigatorias:
        sinalizar(df[c].isna() & ~vazias, f"Campo ausente: {c}")
    for c in ["cpu_usage", "memory_usage", "utilization"]:
        sinalizar(df[c].notna() & ~df[c].between(0, 100), f"Percentual fora de 0–100: {c}")
    for c in ["net_io", "disk_io", "price_per_hour", "latency_ms", "throughput", "cost"]:
        sinalizar(df[c] < 0, f"Valor negativo: {c}")
    for c in ["vcpu", "ram_gb"]:
        sinalizar(df[c] <= 0, f"Capacidade não positiva: {c}")
    sinalizar(df.vcpu.notna() & (df.vcpu % 1 != 0), "vCPU deve ser inteiro")
    sinalizar(df.cloud_provider.notna() & ~df.cloud_provider.isin(["AWS", "Azure", "GCP"]),
              "Provedor não reconhecido nesta base")
    sinalizar(df.target.notna() & ~df.target.isin(["scale_up", "scale_down", "no_action"]),
              "Ação target não reconhecida")
    # Não deduplicar apenas por timestamp: pode haver máquinas diferentes no mesmo instante.
    candidatas = df.loc[motivos.eq("")]
    sinalizar(df.index.isin(candidatas.index[candidatas.duplicated(keep="first")]),
              "Registro duplicado após conversões")
    rejeitadas = df.loc[motivos.ne("")].copy()
    rejeitadas["motivo_revisao"] = motivos.loc[rejeitadas.index].str.rstrip("; ")
    limpa = df.loc[motivos.eq("")].sort_values("timestamp", kind="stable").copy()
    limpa["vcpu"] = limpa.vcpu.astype("Int64")
    resumo = pd.DataFrame({"Indicador": ["Registros originais", "Linhas totalmente vazias",
        "Duplicados após ajuste de espaços", "Registros separados para revisão",
        "Registros disponíveis para análise", "Campos ausentes antes das conversões"],
        "Quantidade": [len(base_original), int(vazias.sum()), int(duplicadas.sum()),
                       len(rejeitadas), len(limpa), int(antes.sum())]})
    qualidade = pd.DataFrame({"ausentes_antes_conversao": antes,
        "ausentes_apos_conversao": df.isna().sum(),
        "falhas_conversao": pd.Series(falhas)}).fillna(0).astype(int)
    return limpa, rejeitadas, resumo, qualidade


In [4]:
def executar_limpeza(_=None):
    global df, df_limpo, registros_revisao, resumo_limpeza, qualidade_colunas
    df_limpo, registros_revisao, resumo_limpeza, qualidade_colunas = limpar_base(df_original)
    df = df_limpo.copy(deep=True)
    display(resumo_limpeza)
    display(qualidade_colunas)
    if not registros_revisao.empty:
        display(registros_revisao.head(10))
    if df.empty:
        raise ValueError('Nenhum registro válido. Revise o arquivo.')
    print('Base limpa em df; original preservado.')
if widgets is not None:
    botao_limpar = widgets.Button(description='Limpar base', button_style='success')
    saida_limpeza = widgets.Output()
    def clicar_limpeza(evento=None):
        with saida_limpeza:
            saida_limpeza.clear_output(wait=True)
            executar_limpeza()
    botao_limpar.on_click(clicar_limpeza)
    display(botao_limpar, saida_limpeza)
    clicar_limpeza()
else:
    executar_limpeza()


Button(button_style='success', description='Limpar base', style=ButtonStyle())

Output()

**3. Visão geral dos dados**

Resumimos utilização, latência e preços. Cada linha é uma observação, não uma máquina ou cobrança independente. RAM_GB é memória; disk_io não é capacidade de armazenamento. As unidades de I/O e a fórmula de cost precisam de confirmação.

In [5]:
print("Registros válidos:", len(df))
print("Período:", df.timestamp.min(), "até", df.timestamp.max())
visao_provedores = df.groupby("cloud_provider").agg(
    registros=("timestamp", "size"), cpu_media=("cpu_usage", "mean"),
    memoria_media=("memory_usage", "mean"), latencia_media_ms=("latency_ms", "mean"),
    preco_medio_hora=("price_per_hour", "mean"))
display(visao_provedores)
custos_base = df.groupby("vm_type").agg(
    registros=("timestamp", "size"), preco_medio_hora=("price_per_hour", "mean"),
    preco_minimo_hora=("price_per_hour", "min"), preco_maximo_hora=("price_per_hour", "max"),
    custo_informado_medio=("cost", "mean"))
display(custos_base)

Registros válidos: 1000
Período: 2024-01-01 00:00:00 até 2024-01-04 11:15:00


,registros,cpu_media,memoria_media,latencia_media_ms,preco_medio_hora
cloud_provider,,,,,
AWS,323,54.987,56.636,243.846,0.025
Azure,326,52.555,53.103,247.723,0.023
GCP,351,53.582,53.833,246.256,0.024


,registros,preco_medio_hora,preco_minimo_hora,preco_maximo_hora,custo_informado_medio
vm_type,,,,,
b1s,351,0.012,0.012,0.012,0.018
n1-standard-1,340,0.047,0.048,0.048,0.137
t2.micro,309,0.012,0.012,0.012,0.017


**4. Alertas operacionais**

CPU/memória acima de 80% ou latência acima de 250 ms. A proporção de observações com alerta representa exposição operacional, não probabilidade de falha. A latência não mede o prazo de abertura.

In [6]:
limite_cpu = 80.0
limite_memoria = 80.0
limite_latencia_ms = 250.0

df["alerta_cpu"] = df.cpu_usage > limite_cpu
df["alerta_memoria"] = df["memory_usage"] > limite_memoria
df["alerta_latencia"] = df.latency_ms > limite_latencia_ms
df["alerta_operacional"] = df[["alerta_cpu", "alerta_memoria", "alerta_latencia"]].any(axis=1)
alertas = pd.DataFrame({"Indicador": ["CPU acima do limite", "Memória acima do limite",
    "Latência acima do limite", "Pelo menos um alerta"],
    "Percentual_observacoes": [100*df[c].mean() for c in
     ["alerta_cpu", "alerta_memoria", "alerta_latencia", "alerta_operacional"]]})
display(alertas)
lista_atencao = df.loc[df.alerta_operacional].sort_values(
    ["cpu_usage", "memory_usage"], ascending=False).head(15)
display(lista_atencao[["timestamp", "cloud_provider", "vm_type", "cpu_usage",
                       "memory_usage", "latency_ms", "target"]])

,Indicador,Percentual_observacoes
0,CPU acima do limite,20.8
1,Memória acima do limite,22.0
2,Latência acima do limite,43.3
3,Pelo menos um alerta,76.6


,timestamp,cloud_provider,vm_type,cpu_usage,memory_usage,latency_ms,target
registro_origem,,,,,,,
223,2024-01-01 18:30:00,Azure,b1s,99.95,14.34,246.07,scale_up
297,2024-01-02 00:40:00,Azure,b1s,99.85,93.98,215.3,scale_up
290,2024-01-02 00:05:00,AWS,n1-standard-1,99.67,43.15,230.84,scale_up
894,2024-01-04 02:25:00,GCP,b1s,99.66,52.71,228.03,scale_up
246,2024-01-01 20:25:00,GCP,n1-standard-1,99.47,69.57,208.43,scale_up
433,2024-01-02 12:00:00,GCP,n1-standard-1,99.39,14.04,257.48,scale_up
570,2024-01-02 23:25:00,AWS,b1s,99.11,38.07,215.02,scale_up
675,2024-01-03 08:10:00,GCP,t2.micro,99.06,71.59,224.45,scale_up
367,2024-01-02 06:30:00,GCP,t2.micro,98.92,50.86,220.37,scale_up


**5. Premissas editáveis**

Capacidade, máquinas, equipe e cronograma são premissas. A capacidade lógica é comparada à demanda com margem; o custo de armazenamento inclui todas as cópias.

**Egress:** volume de saída por mês × tarifa por GB, descontada a franquia assumida. A tarifa inicial de 0,05 U.M./GB é fictícia e editável, não vem do Kaggle. Não converter net_io em GB sem confirmar unidade e janela temporal. A antiga transferência e o egress são o mesmo componente; não são somados em duplicidade.

O multiplicador de demanda afeta armazenamento, carga de CPU/memória, migração e egress. Capacidade e receita ficam constantes para revelar a pressão da demanda inesperada.

In [7]:
#@title Premissas editáveis do projeto
maquinas_referencia = 20 #@param {type:"integer"}
crescimento_demanda_pct = 25 #@param {type:"number"}
armazenamento_demandado_gb = 15000 #@param {type:"number"}
capacidade_referencia_gb = 20000 #@param {type:"number"}
copias_armazenamento = 2 #@param {type:"integer"}
tarifa_armazenamento_gb_mes = 0.02 #@param {type:"number"}
transferencia_demandada_gb_mes = 1000 #@param {type:"number"}
tarifa_transferencia_gb = 0.05 #@param {type:"number"}
horas_operacao_mes = 730 #@param {type:"number"}
custo_implantacao_fixo = 40000 #@param {type:"number"}
custo_configuracao_por_maquina = 100 #@param {type:"number"}
custo_migracao_por_gb = 0.02 #@param {type:"number"}
contingencia_pct = 10 #@param {type:"number"}
custo_equipe_mes = 5000 #@param {type:"number"}
outros_custos_mes = 1000 #@param {type:"number"}
receita_incremental_mes = 12000 #@param {type:"number"}
orcamento_inicial = 65000 #@param {type:"number"}
limite_custo_mensal = 12000 #@param {type:"number"}
limite_payback_anos = 5 #@param {type:"number"}
prazo_desejado_dias = 60 #@param {type:"integer"}
limite_alertas_operacionais_pct = 35 #@param {type:"number"}
margem_armazenamento_pct = 10 #@param {type:"number"}
dias_planejamento = 5 #@param {type:"integer"}
dias_contratacao = 7 #@param {type:"integer"}
dias_seguranca = 10 #@param {type:"integer"}
dias_configuracao_referencia = 8 #@param {type:"integer"}
dias_testes = 7 #@param {type:"integer"}
dias_treinamento = 5 #@param {type:"integer"}
velocidade_migracao_gb_dia = 2000 #@param {type:"number"}

premissas = {k: globals()[k] for k in [
    "maquinas_referencia", "crescimento_demanda_pct", "armazenamento_demandado_gb",
    "capacidade_referencia_gb", "copias_armazenamento", "tarifa_armazenamento_gb_mes",
    "transferencia_demandada_gb_mes", "tarifa_transferencia_gb", "horas_operacao_mes",
    "custo_implantacao_fixo", "custo_configuracao_por_maquina", "custo_migracao_por_gb",
    "contingencia_pct", "custo_equipe_mes", "outros_custos_mes", "receita_incremental_mes",
    "orcamento_inicial", "limite_custo_mensal", "limite_payback_anos", "prazo_desejado_dias",
    "limite_alertas_operacionais_pct", "margem_armazenamento_pct", "dias_planejamento",
    "dias_contratacao", "dias_seguranca", "dias_configuracao_referencia", "dias_testes",
    "dias_treinamento", "velocidade_migracao_gb_dia"]}

# Novas premissas (também disponíveis nos controles do painel)
multiplicador_demanda = 1.0 #@param {type:"number"}
franquia_egress_gb_mes = 0 #@param {type:"number"}
atraso_contratacao_dias = 0 #@param {type:"integer"}
premissas.update(multiplicador_demanda=multiplicador_demanda,
    franquia_egress_gb_mes=franquia_egress_gb_mes, atraso_contratacao_dias=atraso_contratacao_dias)
display(pd.Series(premissas, name='Valor').rename_axis('Premissa').to_frame())


,Valor
Premissa,
maquinas_referencia,20.00
crescimento_demanda_pct,25.00
armazenamento_demandado_gb,15000.00
capacidade_referencia_gb,20000.00
copias_armazenamento,2.00
tarifa_armazenamento_gb_mes,0.02
transferencia_demandada_gb_mes,1000.00
tarifa_transferencia_gb,0.05
horas_operacao_mes,730.00


**6. Simulação de custo, prazo e retorno**

**- Computação mensal:** preço médio/hora x máquinas x horas/mês.

**- Investimento:** implantação + configuração + migração + contingência.

**- Payback:** investimento ÷ caixa líquido anual (receita menos operação). Sem caixa positivo, não há retorno nesse cenário constante.

**- Cronograma:** contratação e segurança em paralelo; configuração; migração e treinamento em paralelo; testes finais.

CPU e memória são projetadas proporcionalmente ao crescimento e à capacidade. Essa aproximação não é uma previsão treinada. Mantemos a latência observada. O payback não inclui impostos ou desconto temporal.

In [10]:
FATORES_CENARIOS = {'Enxuto': 1.0, 'Intermediário': 1.5, 'Ampliado': 2.0}

def validar_premissas(p):
    for nome, valor in p.items():
        if not np.isfinite(valor) or valor < 0:
            raise ValueError(f'{nome}: use um número finito não negativo.')
    positivos = ['maquinas_referencia', 'capacidade_referencia_gb', 'copias_armazenamento',
        'horas_operacao_mes', 'velocidade_migracao_gb_dia', 'multiplicador_demanda',
        'orcamento_inicial', 'limite_custo_mensal', 'prazo_desejado_dias', 'limite_payback_anos']
    for nome in positivos:
        if p[nome] <= 0:
            raise ValueError(f'{nome} deve ser maior que zero.')
    inteiros = ['maquinas_referencia', 'copias_armazenamento', 'dias_planejamento',
        'dias_contratacao', 'dias_seguranca', 'dias_configuracao_referencia',
        'dias_testes', 'dias_treinamento', 'prazo_desejado_dias', 'atraso_contratacao_dias']
    for nome in inteiros:
        if p[nome] % 1 != 0:
            raise ValueError(f'{nome} deve ser inteiro.')
    if p['limite_alertas_operacionais_pct'] > 100:
        raise ValueError('O limite de alertas deve estar entre 0 e 100%.')
    if not (0 < limite_cpu <= 100 and 0 < limite_memoria <= 100 and limite_latencia_ms > 0):
        raise ValueError('Revise os limites operacionais da etapa 4.')

def montar_cronograma(p, fator):
    crescimento = (1 + p['crescimento_demanda_pct'] / 100) * p['multiplicador_demanda']
    configuracao = math.ceil(p['dias_configuracao_referencia'] * fator)
    migracao = math.ceil(p['armazenamento_demandado_gb'] * crescimento / p['velocidade_migracao_gb_dia'])
    tarefas = [
        ('Planejamento', p['dias_planejamento'], []),
        ('Contratação', p['dias_contratacao'] + p['atraso_contratacao_dias'], ['Planejamento']),
        ('Preparação de segurança', p['dias_seguranca'], ['Planejamento']),
        ('Configuração', configuracao, ['Contratação', 'Preparação de segurança']),
        ('Migração', migracao, ['Configuração']),
        ('Treinamento', p['dias_treinamento'], ['Configuração']),
        ('Testes finais', p['dias_testes'], ['Migração', 'Treinamento'])]
    fim, linhas = {}, []
    for etapa, duracao, dependencias in tarefas:
        inicio = max([fim[d] for d in dependencias], default=0)
        fim[etapa] = inicio + duracao
        linhas.append({'Etapa': etapa, 'Inicio_dia': inicio, 'Duracao_dias': duracao,
                       'Fim_dia': fim[etapa], 'Depende_de': ', '.join(dependencias)})
    return pd.DataFrame(linhas)

def simular_cenario(nome, fator, p, base):
    crescimento = (1 + p['crescimento_demanda_pct'] / 100) * p['multiplicador_demanda']
    maquinas = math.ceil(p['maquinas_referencia'] * fator)
    fator_real = maquinas / p['maquinas_referencia']
    cpu = base.cpu_usage * crescimento / fator_real
    memoria = base['memory_usage'] * crescimento / fator_real
    alertas = (cpu > limite_cpu) | (memoria > limite_memoria) | (base.latency_ms > limite_latencia_ms)
    capacidade = p['capacidade_referencia_gb'] * fator
    demanda = p['armazenamento_demandado_gb'] * crescimento
    necessaria = demanda * (1 + p['margem_armazenamento_pct'] / 100)
    computacao = base.price_per_hour.mean() * maquinas * p['horas_operacao_mes']
    armazenamento = capacidade * p['copias_armazenamento'] * p['tarifa_armazenamento_gb_mes']
    volume_egress = p['transferencia_demandada_gb_mes'] * crescimento
    egress_cobrado = max(volume_egress - p['franquia_egress_gb_mes'], 0)
    egress = egress_cobrado * p['tarifa_transferencia_gb']
    operacional = computacao + armazenamento + egress + p['custo_equipe_mes'] + p['outros_custos_mes']
    investimento = (p['custo_implantacao_fixo'] + maquinas * p['custo_configuracao_por_maquina']
                    + demanda * p['custo_migracao_por_gb']) * (1 + p['contingencia_pct'] / 100)
    caixa_anual = (p['receita_incremental_mes'] - operacional) * 12
    payback = investimento / caixa_anual if caixa_anual > 0 else np.inf
    prazo = float(montar_cronograma(p, fator_real).Fim_dia.max())
    financeiros, motivos = [], []
    if investimento > p['orcamento_inicial']: financeiros.append('Investimento acima do orçamento')
    if operacional > p['limite_custo_mensal']: financeiros.append('Custo mensal acima do limite')
    if caixa_anual <= 0: financeiros.append('Caixa incremental não positivo')
    motivos.extend(financeiros)
    if prazo > p['prazo_desejado_dias']: motivos.append('Prazo superior ao desejado')
    if necessaria > capacidade: motivos.append('Armazenamento insuficiente com margem')
    if 100 * alertas.mean() > p['limite_alertas_operacionais_pct']:
        motivos.append('Exposição operacional acima do limite')
    if caixa_anual > 0 and payback > p['limite_payback_anos']:
        motivos.append('Payback acima do limite')
    recomendacao = 'PROJETO INVIÁVEL' if financeiros else ('REVISAR PROJETO' if motivos else 'IMPLEMENTAR')
    return {'Cenario': nome, 'Fator_capacidade': fator_real, 'Maquinas': maquinas,
        'Capacidade_logica_GB': capacidade, 'Demanda_projetada_GB': demanda,
        'Deficit_armazenamento_GB': max(necessaria - capacidade, 0),
        'Computacao_mes_UM': computacao, 'Armazenamento_mes_UM': armazenamento,
        'Egress_volume_GB_mes': volume_egress, 'Egress_cobrado_GB_mes': egress_cobrado,
        'Egress_mes_UM': egress, 'Equipe_mes_UM': p['custo_equipe_mes'],
        'Outros_mes_UM': p['outros_custos_mes'], 'Custo_mensal_UM': operacional,
        'Investimento_UM': investimento, 'Caixa_liquido_anual_UM': caixa_anual,
        'Payback_anos': payback, 'Prazo_dias': prazo, 'Alertas_operacionais_pct': 100 * alertas.mean(),
        'Folga_orcamento_UM': p['orcamento_inicial'] - investimento,
        'Folga_mensal_UM': p['limite_custo_mensal'] - operacional,
        'Custo_total_12_meses_UM': investimento + 12 * operacional,
        'Recomendacao': recomendacao, 'Restricoes_descumpridas': len(motivos),
        'Motivos': '; '.join(motivos) if motivos else 'Todos os critérios atendidos'}

def simular_projeto(p, base):
    validar_premissas(p)
    if base.empty: raise ValueError('A base limpa está vazia.')
    return pd.DataFrame([simular_cenario(n, f, p, base) for n, f in FATORES_CENARIOS.items()])

def selecionar_cenario(resultados):
    aprovados = resultados[resultados.Recomendacao.eq('IMPLEMENTAR')]
    if not aprovados.empty:
        return aprovados.sort_values('Custo_total_12_meses_UM').iloc[0]
    revisaveis = resultados[resultados.Recomendacao.eq('REVISAR PROJETO')]
    referencia = revisaveis if not revisaveis.empty else resultados
    return referencia.sort_values(['Restricoes_descumpridas', 'Custo_total_12_meses_UM']).iloc[0]

def analisar_sensibilidade(p, base):
    choques = [('Base', 1, 1, 1, 0), ('Equipe +15%', 1.15, 1, 1, 0),
        ('Demanda ×2', 1, 2, 1, 0), ('Tarifa egress +50%', 1, 1, 1.5, 0),
        ('Contratação +15 dias', 1, 1, 1, 15), ('Choques combinados', 1.15, 2, 1.5, 15)]
    referencia = simular_projeto(p, base).set_index('Cenario')
    resultados = []
    for choque, equipe, demanda, tarifa, atraso in choques:
        ajustadas = dict(p)
        ajustadas['custo_equipe_mes'] *= equipe
        ajustadas['multiplicador_demanda'] *= demanda
        ajustadas['tarifa_transferencia_gb'] *= tarifa
        ajustadas['atraso_contratacao_dias'] += atraso
        calculados = simular_projeto(ajustadas, base)
        calculados.insert(0, 'Teste', choque)
        calculados['Delta_custo_mensal_UM'] = [r.Custo_mensal_UM - referencia.loc[r.Cenario, 'Custo_mensal_UM']
                                               for r in calculados.itertuples()]
        calculados['Folga_mensal_pct'] = 100 * calculados.Folga_mensal_UM / p['limite_custo_mensal']
        resultados.append(calculados)
    return pd.concat(resultados, ignore_index=True)

def gerar_conclusao(resultados, sens, p):
    s = selecionar_cenario(resultados)
    aprovados = resultados[resultados.Recomendacao.eq('IMPLEMENTAR')]
    if not aprovados.empty:
        abertura = f'IMPLEMENTAR: {s.Cenario} atende aos critérios configurados.'
    elif resultados.Recomendacao.eq('PROJETO INVIÁVEL').all():
        abertura = 'PROJETO INVIÁVEL nas premissas atuais: todos os cenários descumprem pelo menos um critério financeiro.'
    else:
        abertura = f'REVISAR PROJETO: nenhum cenário aprovado; {s.Cenario} é a referência para revisão.'
    texto = [abertura,
        f'{s.Cenario}: investimento de {s.Investimento_UM:,.2f} U.M.; operação de {s.Custo_mensal_UM:,.2f} U.M./mês, '
        f'incluindo egress de {s.Egress_mes_UM:,.2f} U.M./mês; prazo de {s.Prazo_dias:.0f} dias.',
        f'Folga inicial: {s.Folga_orcamento_UM:,.2f} U.M.; folga mensal: {s.Folga_mensal_UM:,.2f} U.M.',
        (f'Payback simples de {s.Payback_anos:.2f} anos.' if np.isfinite(s.Payback_anos)
         else 'Sem payback simples finito: caixa incremental não positivo.'),
        f'Exposição operacional: {s.Alertas_operacionais_pct:.1f}% de observações.']
    for r in resultados.itertuples():
        texto.append(f'{r.Cenario}: {r.Recomendacao} — {r.Motivos}.')
    sal = sens[(sens.Teste == 'Equipe +15%') & (sens.Cenario == s.Cenario)].iloc[0]
    dob = sens[(sens.Teste == 'Demanda ×2') & (sens.Cenario == s.Cenario)].iloc[0]
    texto.append(f'Em {s.Cenario}, equipe +15% acrescenta {sal.Delta_custo_mensal_UM:,.2f} U.M./mês '
        f'e resulta em {sal.Recomendacao}; demanda dobrada gera déficit de {dob.Deficit_armazenamento_GB:,.0f} GB '
        f'com margem e resulta em {dob.Recomendacao}.')
    if not aprovados.empty and (sens[sens.Cenario.eq(s.Cenario)].Recomendacao != 'IMPLEMENTAR').any():
        texto.append('A aprovação no caso base não se mantém em todos os testes de sensibilidade; revisar os choques antes do piloto.')
    texto.append('Confirmar demanda, moeda, tarifas, receita e prazos com a empresa antes de aprovar a expansão. '
                 'A viabilidade é condicionada às premissas e não foi validada em produção.')
    return '\n\n'.join(texto)

colunas_sensibilidade = ['Teste', 'Cenario', 'Custo_mensal_UM', 'Delta_custo_mensal_UM',
    'Folga_mensal_UM', 'Folga_mensal_pct', 'Deficit_armazenamento_GB', 'Prazo_dias', 'Recomendacao']

def calcular_estado(p):
    resultados = simular_projeto(p, df)
    sens = analisar_sensibilidade(p, df)
    return {'valido': True, 'premissas': dict(p), 'cenarios': resultados,
            'sensibilidade': sens, 'conclusao': gerar_conclusao(resultados, sens, p)}

cenarios = simular_projeto(premissas, df)
selecionado = selecionar_cenario(cenarios)
display(cenarios[['Cenario', 'Investimento_UM', 'Custo_mensal_UM', 'Egress_mes_UM',
                  'Payback_anos', 'Prazo_dias', 'Recomendacao', 'Motivos']])

,Cenario,Investimento_UM,Custo_mensal_UM,Egress_mes_UM,Payback_anos,Prazo_dias,Recomendacao,Motivos
0,Enxuto,46612.5,7212.117,62.5,0.811,40.0,REVISAR PROJETO,Armazenamento insuficiente com margem; Exposição operacional acima do limite
1,Intermediário,47712.5,7786.926,62.5,0.944,44.0,REVISAR PROJETO,Exposição operacional acima do limite
2,Ampliado,48812.5,8361.735,62.5,1.118,48.0,REVISAR PROJETO,Exposição operacional acima do limite


**7. Sensibilidade financeira e de projeto**

Testar equipe +15%, demanda ×2, tarifa de egress +50%, contratação +15 dias e choques combinados. São testes determinísticos, não probabilidades de ocorrência. Manter capacidade e receita constantes evidencia déficits e pressão sobre o caixa.

Folga negativa indica estouro. A folga percentual do custo mensal é a distância até o limite dividida pelo limite. Dependências do Gantt podem absorver parte de um atraso na contratação. A tabela inicial abaixo será recalculada no painel da etapa 8 quando os controles forem alterados.

In [11]:
sensibilidade = analisar_sensibilidade(premissas, df)
display(sensibilidade[colunas_sensibilidade].round(2))

,Teste,Cenario,Custo_mensal_UM,Delta_custo_mensal_UM,Folga_mensal_UM,Folga_mensal_pct,Deficit_armazenamento_GB,Prazo_dias,Recomendacao
0,Base,Enxuto,7212.12,0.00,4787.88,39.90,625.0,40.0,REVISAR PROJETO
1,Base,Intermediário,7786.93,0.00,4213.07,35.11,0.0,44.0,REVISAR PROJETO
2,Base,Ampliado,8361.73,0.00,3638.27,30.32,0.0,48.0,REVISAR PROJETO
3,Equipe +15%,Enxuto,7962.12,750.00,4037.88,33.65,625.0,40.0,REVISAR PROJETO
4,Equipe +15%,Intermediário,8536.93,750.00,3463.07,28.86,0.0,44.0,REVISAR PROJETO
5,Equipe +15%,Ampliado,9111.73,750.00,2888.27,24.07,0.0,48.0,REVISAR PROJETO
6,Demanda ×2,Enxuto,7274.62,62.50,4725.38,39.38,21250.0,49.0,REVISAR PROJETO
7,Demanda ×2,Intermediário,7849.43,62.50,4150.57,34.59,11250.0,53.0,REVISAR PROJETO
8,Demanda ×2,Ampliado,8424.23,62.50,3575.77,29.80,1250.0,57.0,REVISAR PROJETO
9,Tarifa egress +50%,Enxuto,7243.37,31.25,4756.63,39.64,625.0,40.0,REVISAR PROJETO


**8. Painel e cronograma**

O cronograma mostra as dependências entre atividades





In [12]:
def desenhar_painel(estado, cenario_gantt='Automático'):
    resultados, p, sens = estado['cenarios'], estado['premissas'], estado['sensibilidade']
    fig, axs = plt.subplots(2, 2, figsize=(12, 8))
    for ax, coluna, titulo, unidade, limite in [
        (axs[0, 0], 'Investimento_UM', 'Investimento inicial', 'U.M.', p['orcamento_inicial']),
        (axs[0, 1], 'Custo_mensal_UM', 'Operação mensal (inclui egress)', 'U.M./mês', p['limite_custo_mensal']),
        (axs[1, 0], 'Prazo_dias', 'Prazo de implantação', 'Dias', p['prazo_desejado_dias']),
        (axs[1, 1], 'Alertas_operacionais_pct', 'Exposição operacional', '% de observações', p['limite_alertas_operacionais_pct'])]:
        ax.bar(resultados.Cenario, resultados[coluna], color=['#2563eb', '#0f766e', '#7c3aed'])
        ax.axhline(limite, color='#b91c1c', linestyle='--', label='Limite')
        ax.set(title=titulo, ylabel=unidade)
        ax.set_ylim(0, max(float(resultados[coluna].max()), limite, 1) * 1.2)
        ax.grid(axis='y', alpha=.2); ax.legend(fontsize=8)
    fig.tight_layout(); plt.show()
    linha = (selecionar_cenario(resultados) if cenario_gantt == 'Automático' else
             resultados[resultados.Cenario.eq(cenario_gantt)].iloc[0])
    cronograma = montar_cronograma(p, linha.Fator_capacidade)
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.barh(cronograma.Etapa, cronograma.Duracao_dias, left=cronograma.Inicio_dia, color='#0f766e')
    ax.invert_yaxis()
    ax.axvline(p['prazo_desejado_dias'], color='#b91c1c', linestyle='--', label='Prazo desejado')
    ax.set(title=f'Cronograma — {linha.Cenario} ({linha.Prazo_dias:.0f} dias)', xlabel='Dias corridos desde o início')
    ax.grid(axis='x', alpha=.2); ax.legend()
    fig.tight_layout(); plt.show()
    display(cronograma)
    ordem = ['Base', 'Equipe +15%', 'Demanda ×2', 'Tarifa egress +50%', 'Contratação +15 dias', 'Choques combinados']
    comparacao = sens.pivot(index='Teste', columns='Cenario', values='Custo_mensal_UM').reindex(ordem)
    comparacao.plot.bar(figsize=(12, 5), rot=18, title='Sensibilidade do custo mensal')
    plt.axhline(p['limite_custo_mensal'], color='#b91c1c', linestyle='--', label='Limite mensal')
    plt.ylabel('U.M./mês'); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

def mostrar_estado(estado, cenario_gantt='Automático'):
    display(estado['cenarios'][['Cenario', 'Investimento_UM', 'Custo_mensal_UM', 'Egress_mes_UM',
        'Folga_orcamento_UM', 'Folga_mensal_UM', 'Payback_anos', 'Recomendacao', 'Motivos']].round(2))
    display(estado['sensibilidade'][colunas_sensibilidade].round(2))
    desenhar_painel(estado, cenario_gantt)
    print(estado['conclusao'])

estado_painel = {}
premissas_iniciais = dict(premissas)

if widgets is not None:
    estilo = {'description_width': '190px'}
    largura = widgets.Layout(width='95%')
    controles = {
        'multiplicador_demanda': widgets.FloatSlider(description='Demanda (multiplicador)',
            value=premissas['multiplicador_demanda'], min=min(.5, premissas['multiplicador_demanda']),
            max=max(3, premissas['multiplicador_demanda']), step=.1, continuous_update=False, style=estilo, layout=largura),
        'custo_equipe_mes': widgets.FloatSlider(description='Equipe (U.M./mês)',
            value=premissas['custo_equipe_mes'], min=0, max=max(15000, premissas['custo_equipe_mes'] * 3),
            step=50, continuous_update=False, style=estilo, layout=largura),
        'atraso_contratacao_dias': widgets.IntSlider(description='Atraso contratação (dias)',
            value=premissas['atraso_contratacao_dias'], min=0, max=max(60, premissas['atraso_contratacao_dias']),
            continuous_update=False, style=estilo, layout=largura)}
    for chave, descricao in [
        ('tarifa_transferencia_gb', 'Egress (U.M./GB)'), ('transferencia_demandada_gb_mes', 'Saída base (GB/mês)'),
        ('franquia_egress_gb_mes', 'Franquia egress (GB/mês)'), ('orcamento_inicial', 'Orçamento inicial (U.M.)'),
        ('limite_custo_mensal', 'Limite mensal (U.M.)'), ('receita_incremental_mes', 'Receita (U.M./mês)'),
        ('capacidade_referencia_gb', 'Capacidade base (GB)'), ('velocidade_migracao_gb_dia', 'Migração (GB/dia)')]:
        controles[chave] = widgets.FloatText(description=descricao, value=premissas[chave], style=estilo, layout=largura)
    controles['maquinas_referencia'] = widgets.IntText(description='Máquinas de referência',
        value=premissas['maquinas_referencia'], style=estilo, layout=largura)
    controle_gantt = widgets.Dropdown(description='Cenário do Gantt',
        options=['Automático'] + list(FATORES_CENARIOS), style=estilo, layout=largura)
    botao_restaurar = widgets.Button(description='Restaurar premissas', button_style='info')
    saida_painel = widgets.Output()
    restaurando = False

    def atualizar_painel(evento=None):
        global estado_painel
        if restaurando:
            return
        p = dict(premissas_iniciais)
        p.update({nome: controle.value for nome, controle in controles.items()})
        with saida_painel:
            saida_painel.clear_output(wait=True)
            try:
                estado_painel = calcular_estado(p)
                mostrar_estado(estado_painel, controle_gantt.value)
            except ValueError as erro:
                estado_painel = {'valido': False, 'conclusao': f'Premissas inválidas: {erro}. Corrija os campos para recalcular.'}
                print(estado_painel['conclusao'])
        if 'saida_conclusao' in globals():
            with saida_conclusao:
                saida_conclusao.clear_output(wait=True)
                print(estado_painel['conclusao'])

    def restaurar_premissas(evento=None):
        global restaurando
        restaurando = True
        try:
            for nome, controle in controles.items():
                controle.value = premissas_iniciais[nome]
            controle_gantt.value = 'Automático'
        finally:
            restaurando = False
        atualizar_painel()

    for controle in list(controles.values()) + [controle_gantt]:
        controle.observe(atualizar_painel, names='value')
    botao_restaurar.on_click(restaurar_premissas)
    display(widgets.VBox(list(controles.values()) + [controle_gantt, botao_restaurar]), saida_painel)
    atualizar_painel()
else:
    estado_painel = calcular_estado(premissas)
    mostrar_estado(estado_painel)
    print('Para usar sliders, abra no Colab com ipywidgets disponível.')


Output()

**9. Planejar a abertura**

Confirmar demanda e tarifas; aprovar cenário e orçamento; contratar e preparar segurança; configurar; migrar em piloto; testar acesso, integridade, desempenho e restauração; iniciar a operação após aprovação dos testes.
Se a latência permanecer acima do limite, investigar rede e região: ampliar máquinas, por si só, não garante menor latência.

**10. Conclusão**

O texto usa os resultados vigentes do painel e é atualizado pelos controles da etapa 8. “Projeto inviável” significa inviável com os limites e premissas configurados, não impossibilidade definitiva da expansão. Sem aprovação, o cenário indicado é apenas referência para revisão.


In [13]:
if widgets is not None:
    saida_conclusao = widgets.Output()
    display(saida_conclusao)
    with saida_conclusao:
        print(estado_painel['conclusao'])
else:
    print(estado_painel['conclusao'])

Output()